In [1]:
!pip install openai pandas pymupdf python-dotenv

In [12]:
import pandas as pd
import json
import os
from openai import OpenAI
import fitz
import base64
from datetime import datetime
import getpass
import time
from dotenv import load_dotenv

In [13]:
load_dotenv()
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY was not found.")
client = OpenAI(api_key=api_key)

In [14]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Reply with exactly this text: API connection successful."}]
)
print(response.choices[0].message.content)

API connection successful.


In [19]:
pdf_path = input("Enter the full path to your PDF file: ").strip().strip('"')

if os.path.exists(pdf_path):
    print(f"File {os.path.basename(pdf_path)} found")
else:
    print("File not found. Please check the file path again.")

Enter the full path to your PDF file:  "C:\Users\tdmur\Downloads\The Scientific World Journal - 2025 - Addis - Intestinal Parasitic Infection  Prevalence and Associated Risk Factors at.pdf"


File The Scientific World Journal - 2025 - Addis - Intestinal Parasitic Infection  Prevalence and Associated Risk Factors at.pdf found


In [27]:
#This code creates relative file paths for outputs of this agent to go to and be stored locally.
output_dir = "data/output/pdf_to_csv/"
base_dir = os.path.dirname(os.path.abspath("__file__"))
input_dir = os.path.join(base_dir, "data", "input")
output_dir = os.path.join(base_dir, "data", "output", "pdf_to_csv")
raw_response_dir = os.path.join(base_dir, "data", "raw_responses")
log_dir = os.path.join(base_dir, "data", "logs")

# Creates all folders if they don't exist
os.makedirs(input_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)
os.makedirs(raw_response_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

print(f"Folders ready:")
print(f"Input: {input_dir}")
print(f"Output: {output_dir}")
print(f"Raw responses: {raw_response_dir}")
print(f"Logs: {log_dir}")
#log entries is a list that populates with the information of each table that is scanned from the pdf. It is crucial because
#it contains information about each table (what issues were flagged, what source are these tables from, when was this extracted, etc.
#table_count counts how many tables are scanned.
log_entries = []
record_counter = 1
pages_scanned = 0
pages_with_tables = 0
table_count = 0

#fitz is a package that facilitates GPT Vision's ability to read images.
pdf_document = fitz.open(pdf_path)
total_pages = len(pdf_document)

#outputs a message telling the user that the agent has successfully opened the pdf, and lets the reader know how many pages
# it detects.
print(f"Opened PDF: {pdf_path}")
print(f"Total Pages: {total_pages}")

#Establishes loop through every page that denotes how many pages scanned, transforms the table images in the PDFs to
#readable images for GPT Vision and creates a clean, workable, and readable response.
for page_num in range(len(pdf_document)):
    page = pdf_document[page_num]
#Converts the page to a readable image for OpenAI API "GPT Vision"    
    mat = fitz.Matrix(2, 2)
    pix = page.get_pixmap(matrix=mat)
    img_bytes = pix.tobytes("png")
    img_base64 = base64.b64encode(img_bytes).decode("utf-8")
    
#Builds a prompt that asks ChatGPT if there are tables in the image.This is done as a variable so its less confusing to look at
#compared to the alternative of putting between a plethora of brackets and parenthesis.
    extraction_prompt = """ You are a highly skilled tool used for data extraction from PDF files.
Carefully examine this PDF page and extract all tables exactly as they appear. 
For each table: 
- Record the table title if visible on the page
- Preserve exact column headers including any units
- Do not merge, split, or summarize any cells
- If a cell appears empty, represent it as an empty string ""
- Flag any cells that appear merged, shifted, or ambiguous
- If a table has subheaders or merged header rows, flatten them into a single header row by combining the parent and child header with a dash. For instance, if a header says something like "IPI's" and it is divided between Positive % and Negative %, divide the subheaders into two different headers, one being IPI's - Positive% and the other being IPI's - Negative%
- Ensure that a distinction is made between how subheaders and additional context for values are treated based on instructions above. 
- If you are unsure about a merged cell, duplicate the parent label for each child column rather than leaving it empty
- If a page contains multiple tables, extract them one at a time and include each as a separate entry in the tables array
- Focus on accuracy over speed

For statistical and complex values:
- Some columns may contain complex values that include multiple numbers, brackets, and commas within a single cell. A common example is statistical values like odds ratios which look like "3.046 (0.685, 5.596), 0.016". This entire string is one value and must stay in a single cell. 
- Never split on commas that appear inside parentheses or brackets.
- General rule: if you see a pattern like "number (number, number), number" treat the whole thing as one cell value regardless of the commas inside it.
- When a table has a column that visually contains both a ratio/range AND a p-value separated by a comma, keep them together as one string in one cell.
- Never create more columns than there are headers. If you find yourself with more values than headers in a row, you have split a cell that should have stayed together. Go back and merge the split values until the row length matches the header count.
- After extracting each row, count its values and compare to the number of headers. If they do not match, find which cell was incorrectly split and fix it before moving on.
- For merged cells that span multiple rows in the first column, repeat the value in every row rather than leaving it blank
- For subheaders that span multiple columns, flatten them into combined headers using " - " as a separator. For example if "IPIs" spans "Positive (%)" and "Negative (%)", create headers "IPIs - Positive (%)" and "IPIs - Negative (%)".
- Columns containing statistical results often have a format like "X.XXX (X.XXX, X.XXX), X.XXX" where the entire string including the parentheses and everything inside them is ONE value. The commas inside the parentheses are NOT column separators. Treat the entire string as a single cell value.
- If you are unsure whether a comma is a column separator or part of a value, look at the header count. If splitting on the comma would give you more values than headers, it is part of the value.
- Never leave the first column empty in any row. If a cell visually spans multiple rows in the PDF, repeat its value in every row.
- Here is the universal rule for identifying one cell vs multiple cells:
  - Text in parentheses attached to a number = part of that number's cell
  - A comma between two numbers inside parentheses = part of the same cell
  - A comma between a closing parenthesis and a number = part of the same cell
  - A comma between two standalone numbers with no parentheses = likely a column separator

For footnotes and notes:
- Tables often have footnotes or notes below them such as "Note: 1 = reference value" or "Abbreviation: COR, crude odds ratio". Do not include these as data rows. If you see them, note their presence in Issues but do not extract them as table data.

For tables spanning multiple pages:
- Some tables span multiple pages. If you see a table that appears to continue from the previous page (no title, starts mid-data, or has no headers), note this in Issues as "Table appears to continue from previous page" and set verdict to "Requires Correction". Extract whatever rows are visible and preserve them.
- If you see a table that appears to continue onto the next page (cuts off mid-data), note this in Issues as "Table appears to continue on next page" and still extract all visible rows.

When validating work:
- Check every column for missing or empty values
- Before writing JSON response, count the number of headers and the amount of values in a row. Make sure that they match, if they don't, fix it before responding. This check is mandatory.
- Check that row labels are present and not shifted (in the correct place according to the original material)
- Check that the number of values in EVERY row is equal to the number of headers exactly
- Give a verdict of "Approved" if the table looks clean and complete
- Give a verdict of "Needs Review" if any issues are found, if you had to make assumptions, or if you are uncertain about any cell.
- Based on what you know as A highly skilled tool used for data extraction, make a suggestion in the table about how the user can fix tables that "Require Correction" output it in the suggestions part.
- List issues and suggestions where necessary
- If you are uncertain about any part of the table structure, extract your best attempt, set verdict to "Needs Review", and describe exactly what you were uncertain about in Issues.

Respond ONLY in JSON with no extra text or markdown"
{"tables": [{"title": "table title or empty string", "headers": ["column1", "column2"], "rows": [["value1", "value2"]], "Verdict": "Approved", "Issues": [], "Suggestion": ""}]}"""

#This section of code is when the prompt is sent to GPT. It outlines GPT's role and what it should expect inputs should be.
#the two "content" lines tell GPT that an image is included and the GPT must use GPT vision to carry out the instructions
#given to it by the prompt. 
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=8000,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{img_base64}"
                    }
                },
                { 
                    "type": "text",
                    "text": extraction_prompt
                }
            ]
        }]
    )
    if total_pages > 10:
        time.sleep(2)
    elif total_pages > 20: 
        time.sleep(6)
    elif total_pages > 30:
        time.sleep(10)
        
#This code goes through each JSON formatted GPT response and ensures that a proper, readable JSON structure is in the output. 
#the "Try , except" works as a fail safe to essentially ignore and move on from the strict JSON instructions for a table if there
# is an error in outputting it in such a way. Thus, the code doesn't crash with a faulty, less clean GPT response.
    clean = response.choices[0].message.content.replace("```json", "").replace("```", "").strip()
    raw_response_path = os.path.join(raw_response_dir, f"page_{page_num+1}.json")
    with open(raw_response_path, "w", encoding = "utf-8") as f:
        json.dump({
            "page" : page_num + 1,
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "raw_response": response.choices[0].message.content,
            "clean_response": clean,
                }, f, indent = 2)
    print(f"\nProcessing page {page_num+1} of {total_pages}...")
    try:
        parsed = json.loads(clean)
    except json.JSONDecodeError:
        print(f"Could not parse response on page {page_num + 1}. Skipping")
        log_entries.append({
            "record_id": f"TOMMY-TBL-{record_counter:03d}",
            "record_type": "table_cell",
            "source_filename": os.path.basename(pdf_path),
            "source_page_number": page_num + 1,
            "claim": f"Parse error found on page {page_num +1}",
            "evidence_snippet": "None",
            "extraction_status": "Failed",
            "structural_status": "",
            "verification_status": "",
            "human_status": "Pending",
            "extracted_by": "PDFtoCSV-v2",
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "output_csv": "None",
            "rows_modified": "No",
            "table_title": "N/A",
            "output_csv": "N/A",
            "issues": "Could not parse GPT response as JSON. Check raw_responses folder.",
            "suggestion": "None"
        })
        continue
#This code creates a dictionary entry for each of the tables that was gathered by GPT and pairs it with the page it was on. 
#It disregards the pages shown to not have tables. It then prints how many tables were found on the pages that included tables and prints their titles.
    tables = parsed.get("tables", [])
    if not tables:
        print(f"No tables found on page {page_num + 1}.")
        log_entries.append({
            "record_id": f"TOMMY-TBL-{record_counter:03d}",
            "record_type": "table_cell",
            "source_filename": os.path.basename(pdf_path),
            "source_page_number": page_num + 1,
            "claim": f"No tables found on page {page_num +1}",
            "evidence_snippet": "None",
            "extraction_status": "Failed",
            "structural_status": "",
            "verification_status": "",
            "human_status": "Pending",
            "extracted_by": "PDFtoCSV-v2",
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "output_csv": "None",
            "rows_modified": "No",
            "table_title": "N/A",
            "output_csv": "N/A",
            "issues": "None",
            "suggestion": "None"
        })
        continue
    else:
        print(f"Tables found: {len(tables)}")
        for table in tables:
            print(f"{table.get('title', 'Untitled table')}")
#This for loop assigns each table in the dictionary of tables to a table ID to be referenced and printed. To ensure each table ID includes the proper information,
#a .get function is used to take that information from the dictionary list and assigns them.
    for table_id, table in enumerate(tables):
        headers = table.get("headers", [])
        rows = table.get("rows", [])
        title = table.get("title", "").strip()
        verdict = table.get("Verdict", "AI Reviewed: Uncertain")
        issues = table.get("Issues", [])
        suggestion = table.get("Suggestion", "")
#This acts as a fail safe, telling the agent to skip tables where it can not decipher if the table has headers or rows.
#It prevents the entire code from crashing because of one bad table. 
        if not headers or not rows:
            print(f"No headers or rows. Skipping.")
            continue
            
#Representing a table in pandas requires that the amount of values in a row matches with the amount of headers for the table.
#This code ensures that this can happen by either adding row spaces to match with the amount of headers or cutting the row short
#to match with the amount of headers. It also denotes that tables that are altered like this require review by the user. 
        row_lengths = [len(row) for row in rows]
        max_row_length = max(row_lengths)
        rows_modified = False
        if max_row_length != len(headers):
            issues.append(f"Row length mismatched with headers. Rows were adjusted to match headers. Manual review recommended")
            verdict = "Requires Correction"
            rows_modified = True
            updated_rows = []
            for row in rows:
                if len(row) < len(headers):
                    updated_rows.append(row + [""] * (len(headers) - len(row)))
                else:
                    updated_rows.append(row[:len(headers)])
            rows = updated_rows
        table_count += 1
#Transforms the information that GPT developed about each table_id into a dataframe that is then turned into a csv file. 
        df = pd.DataFrame(rows, columns = headers)
#Names the datafram that was just created, assigns it to a file path to be found in a certain folder, and officially turns the
#dataframe into a CSV file!
        filename = f"table_p{page_num+1}_t{table_id+1}.csv"
        filepath = os.path.join(output_dir, filename)
        df.to_csv(filepath, index=False)
#Now the CSV file that was just saved is loaded back into pandas to see if it saved correctly. 
#missing function counts all of the empty values in the table. If a table is deemed to be missing a value, then the "verdict"
# says "Requires Correction" (this prompts the user to look over the CSV and see if it is significantly different than the original table.
        df_check = pd.read_csv(filepath, keep_default_na=False)
        missing = df_check.isna().sum().sum()
        if missing > 0:
            issues.append(f"{missing} truly missing values found after re-read")
            verdict = "Requires Correction"
        print(f"\nPage {page_num + 1}, Table {table_id+1}: {verdict}")
 #The title is printed in the table based off of the csv if it can be parsed, to assign the verdict and issues to it.       
        if title:
            print(f"Title: {title}")
# The issues for each of the tables is outputted in the log table, so the user knows what to go back and look at.    
        if issues:
            for issue in issues:
                print(f"Issues: {issue}")
        log_entries.append({
            "record_id": f"TOMMY-TBL-{record_counter:03d}",
            "record_type": "table_cell",
            "source_filename": os.path.basename(pdf_path),
            "source_page_number": page_num + 1,
            "extraction_status": "Extracted" if not issues else "Needs Review",
            "structural_status": "",
            "verification_status": "",
            "human_status": "Pending",
            "extracted_by": "PDFtoCSV-v2",
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "output_csv": filename,
            "rows_modified": "Yes, rows padded or trimmed to match row length with headers" if rows_modified else "No",
            "table_title": title if title else "No title",
            "issues": issues if issues else "None",
            "suggestion": suggestion if suggestion else "None",
        })
    
            
pdf_document.close()

extracted = sum(1 for e in log_entries if e["extraction_status"] == "Extracted")
needs_review = sum(1 for e in log_entries if e["extraction_status"] == "Needs Review")
failed = sum(1 for e in log_entries if e["extraction_status"] == "Failed")

print(f"Extraction complete.")
print(f"Tables extracted: {table_count}")
print(f"Extracted: {extracted}")
print(f"Needs Review {needs_review}")
print(f"Failed: {failed}")
print(f"Important Note: human_status is pending for all tables. Manual review is required before Approved can be set")             


Folders ready:
Input: C:\Users\tdmur\notebooks(research)\data\input
Output: C:\Users\tdmur\notebooks(research)\data\output\pdf_to_csv
Raw responses: C:\Users\tdmur\notebooks(research)\data\raw_responses
Logs: C:\Users\tdmur\notebooks(research)\data\logs
Opened PDF: C:\Users\tdmur\Downloads\The Scientific World Journal - 2025 - Addis - Intestinal Parasitic Infection  Prevalence and Associated Risk Factors at.pdf
Total Pages: 9

Processing page 1 of 9...
Tables found: 1


 Page 1, Table 1: Approved

Processing page 2 of 9...
No tables found on page 2.

Processing page 3 of 9...
Tables found: 1
Sociodemographic characteristics of study participants, Delgi Primary Hospital, Central Gondar, Ethiopia (n = 404)

 Page 3, Table 1: Approved
Title: Sociodemographic characteristics of study participants, Delgi Primary Hospital, Central Gondar, Ethiopia (n = 404)

Processing page 4 of 9...
Tables found: 1
Distribution of intestinal parasite species by sex and age group among study participants (n 

In [23]:
import glob
print("Preview of extracted tables:\n")
csv_files = glob.glob(os.path.join(output_dir, "table_*.csv"))

if not csv_files:
    print("No CSV files found in output folders")
else:
    for csv_file in sorted(csv_files):
        print(f"{os.path.basename(csv_file)}")
        df_preview = pd.read_csv(csv_file, keep_default_na = False)
        display(df_preview.head())
        print()

Preview of extracted tables:

table_p3_t1.csv


,Characteristics,Categories,Frequency (n = 404),Percentage (%)
0,Sex,Male,210,52.0
1,,Female,194,48.0
2,Age group (years),5–14,151,37.4
3,,15–24,151,37.4
4,,Above 25,102,25.2



table_p3_t2.csv


,Unnamed: 0,Characteristics,Categories,Frequency (n = 404),Percentage (%)
0,,Sex,Male,210,52.0
1,,Sex,Female,194,48.0
2,,Age group (years),5–14,151,37.4
3,,Age group (years),15–24,151,37.4
4,,Age group (years),Above 25,102,25.2



table_p4_t1.csv


...

,"Note : N = reference value, N = total.","Abbreviation: AOR, adjusted odds ratio (multivariate regression model).",Statistically significant at p < 0.05.,Unnamed: 3
0,,,,



table_p8_t1.csv


,Unnamed: 0,AOR (95% CI),P-value
0,Handwashing after using the toilet,3.941 (1.619-1.432),0.001
1,Eating unwashed vegetables,,
2,Dirty matter under fingernails,3.934 (1.615-1.417),0.001



table_p9_t1.csv


,Period of growth,Periods of crisis,Average economic growth rate,Average nominal borrowing interest rate,Average ex-post real borrowing interest rate,Inflation,"Gross domestic investment, % of GDP","Gross domestic savings, % of GDP"
0,2002-2007,Foreign-exchange crisis of 2001,6.8%,29.5% (deposits),11.1% (deposits),Down to 8.8% from 45%,Up to 21% from 18%,Down to 15% from 18%
1,2000-2008,Foreign-exchange crises of 1999 and of 2001,3.7%,52.8%,43.1%,Down to 5.7% from 7.0%,18% in 2000 and 2008; upward trend was observed,"Up to 15% from 14%, with an uptrend"
2,1984-2013,Banking crisis of 1981-1984,5.6%,18.2%,8%,Down to 1.8% from 19.9%,Up to 23.8% from 12.4%,Up to 20.6% from 2.3%
3,1980-2013,,6%,14.1%,5.4%,"Around 6% on average (lowest - 0.5%, highest 9...",Up to 33% from 18%,Up to 32% from 20.2%


In [25]:
#prints the log entries, creates the log_entries csv in the same folder as the other tables and tells you if nothing was saved. 
if log_entries:
    log_df = pd.DataFrame(log_entries)
    log_csv_path = os.path.join(log_dir, "extraction_log.csv")
    log_df.to_csv(log_csv_path, index = False)
    print(f"Log saved to {log_csv_path}")

    log_json_path = os.path.join(log_dir, "extraction_log.json")
    log_df.to_json(log_json_path, orient= "records", indent = 2)
    print(f"Log saved to JSON to {log_json_path}")

    display(log_df)
else:
    print("No records for log.")

Log saved to C:\Users\tdmur\notebooks(research)\data\logs\extraction_log.csv
Log saved to JSON to C:\Users\tdmur\notebooks(research)\data\logs\extraction_log.json


,record_id,record_type,source_filename,source_page_number,claim,evidence_snippet,extraction_status,structural_status,verification_status,human_status,extracted_by,timestamp,output_csv,rows_modified,table_title,issues,suggestion
0,TOMMY-TBL-001,table_cell,The Scientific World Journal - 2025 - Addis - ...,1,No tables found on page 1,None,Failed,,,Pending,PDFtoCSV-v2,2026-08-04 09:29:06,N/A,No,N/A,None,None
1,TOMMY-TBL-001,table_cell,The Scientific World Journal - 2025 - Addis - ...,2,No tables found on page 2,None,Failed,,,Pending,PDFtoCSV-v2,2026-08-04 09:29:08,N/A,No,N/A,None,None
2,TOMMY-TBL-001,table_cell,The Scientific World Journal - 2025 - Addis - ...,3,NaN,NaN,Extracted,,,Pending,PDFtoCSV-v2,2026-08-04 09:29:18,table_p3_t1.csv,No,Sociodemographic characteristics of study part...,None,None
3,TOMMY-TBL-001,table_cell,The Scientific World Journal - 2025 - Addis - ...,4,NaN,NaN,Needs Review,,,Pending,PDFtoCSV-v2,2026-08-04 09:29:28,table_p4_t1.csv,"Yes, rows padded or trimmed to match row lengt...",Distribution of intestinal parasite species by...,[Row length mismatched with headers. Rows were...,None
4,TOMMY-TBL-001,table_cell,The Scientific World Journal - 2025 - Addis - ...,5,NaN,NaN,Extracted,,,Pending,PDFtoCSV-v2,2026-08-04 09:29:37,table_p5_t1.csv,No,Bivariate logistic regression analysis of soci...,None,None
5,TOMMY-TBL-001,table_cell,The Scientific World Journal - 2025 - Addis - ...,6,NaN,NaN,Extracted,,,Pending,PDFtoCSV-v2,2026-08-04 09:29:52,table_p6_t1.csv,No,Bivariate logistic regression analysis of life...,None,None
6,TOMMY-TBL-001,table_cell,The Scientific World Journal - 2025 - Addis - ...,6,NaN,NaN,Extracted,,,Pending,PDFtoCSV-v2,2026-08-04 09:29:52,table_p6_t2.csv,No,Bivariate logistic regression analysis of life...,None,None
7,TOMMY-TBL-001,table_cell,The Scientific World Journal - 2025 - Addis - ...,7,NaN,NaN,Needs Review,,,Pending,PDFtoCSV-v2,2026-08-04 09:30:02,table_p7_t1.csv,"Yes, rows padded or trimmed to match row lengt...",Multivariate logistic regression analysis of s...,[Row length mismatched with headers. Rows were...,None
8,TOMMY-TBL-001,table_cell,The Scientific World Journal - 2025 - Addis - ...,7,NaN,NaN,Needs Review,,,Pending,PDFtoCSV-v2,2026-08-04 09:30:02,table_p7_t2.csv,"Yes, rows padded or trimmed to match row lengt...",No title,"[Table appears to continue from previous page,...",[Extract the entire table with context from th...
9,TOMMY-TBL-001,table_cell,The Scientific World Journal - 2025 - Addis - ...,8,NaN,NaN,Extracted,,,Pending,PDFtoCSV-v2,2026-08-04 09:30:05,table_p8_t1.csv,No,No title,None,None


Run Log
Date: 7/24-7/31
Task Performed: Created an AI agent that extracts tables in PDFs to CSV files, reviews each CSV file compared to the original PDF, and informs the user which tables should be reviewed and what the issues are.
Input filename: "C:\Users\tdmur\Downloads\The Scientific World Journal - 2025 - Addis - Intestinal Parasitic Infection  Prevalence and Associated Risk Factors at.pdf"
Notebook or script version: JupyterNotebook
Model: OpenAI GPT 4o Mini
Output filenames: extractionlog.csv, table_P3_t1.csv, table_P4_t1.csv, ..., etc.
Warnings: Does not work very well for tables with subheaders or renamed files. GPT handles things differently each time, more prompting needed.
Execution time: <2 minutes
API cost <$5
Final review/verification: not completely finished, version 1. 

- Pay close attention to parenthesis. If a header includes a value inside parenthesis next to it in the header, include it in the row values in the same format as in the header. For instance N(%) should have two values in each row one which represents the number, which shouldn't have parenthesis around it, and another which represents the percentage, which should be represented with parenthesis. These two values characterise the same value in different ways and should thus be represented in that way. This will usually apply for values that are written on the same horizontal plane.